In [ ]:
!pip install torchtuples pycox
!pip install lifelines

In [ ]:
import pandas as pd
import numpy as np

def encode_columns(column):
    """
    Encodes categorical columns using a predefined mapping strategy.
    Converts 'Positive', 'Yes' to 1, 'Negative', 'No' to 0, and handles missing values.
    """
    column_filled = column.fillna('NA').astype(str)  # Handle missing values
    custom_mapping = {
        'Positive': 1, 'Negative': 0, 'Yes': 1, 'No': 0,
        'Not done': 2, 'NA': -1  # Special handling for missing values
    }
    return column_filled.map(custom_mapping)

def preprocess_features(df, reference_columns=None):
    """
    Preprocesses a DataFrame by handling categorical features, encoding values,
    and filling missing values. Ensures consistency with reference columns.

    Args:
        df (pd.DataFrame): The DataFrame to preprocess.
        reference_columns (list, optional): The list of final columns from training data.

    Returns:
        pd.DataFrame: The processed DataFrame with aligned columns.
    """
    # Apply mappings for specific categorical features
    mappings = {
        'dri_score': {'Low': 1, 'Intermediate': 2, 'Intermediate - TED AML case <missing cytogenetics': 2.5,
                      'High': 3, 'High - TED AML case <missing cytogenetics': 3.5, 'Very high': 4,
                      'N/A - disease not classifiable': -1, 'N/A - non-malignant indication': -2,
                      'N/A - pediatric': -2, 'TBD cytogenetics': -1, 'Missing disease status': -1},
        'psych_disturb': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'cyto_score': {'Favorable': 1, 'Intermediate': 2, 'Normal': 3, 'Other': 4,
                       'Poor': 5, 'TBD': 6, 'Not tested': 6, 'Unknown': 7},
        'diabetes': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'arrhythmia': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'vent_hist': {'No': 0, 'Yes': 1, 'Unknown': 2},
        'renal_issue': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'pulm_severe': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'hepatic_mild': {'No': 0, 'Yes': 1, 'Not done': 2, 'Unknown': 3},
        'cmv_status': {"-/+": 3, "+/-": 2, "-/-": 1, "+/+": 0, "Missing": -1},
        'conditioning_intensity': {'MAC': 5, 'RIC': 4, 'NMA': 3, 'TBD': 2,
                                   'No drugs reported': 1, 'N/A, F(pre-TED) not submitted': 0, 'Missing': -1},
        'cyto_score_detail': {'Poor': 4, 'TBD': 3, 'Not tested': 2, 'Intermediate': 1, 'Favorable': 0, 'Missing': -1},
        'graft_type': {'Bone marrow': 0, 'Peripheral blood': 1},
        'tce_match': {'Permissive': 0, 'Fully matched': 1, 'GvH non-permissive': 2, 'HvG non-permissive': 3, 'NA': -1},
        'sex_match': {'F-F': 1, 'M-M': 2, 'M-F': 3, 'F-M': 4, 'NA': -1}
    }

    # Apply mappings after filling missing values
    for col, mapping in mappings.items():
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').map(mapping)

    # Fill missing numerical values using median
    numerical_cols = [
        'hla_match_c_high', 'hla_high_res_8', 'hla_low_res_6',
        'hla_high_res_6', 'hla_high_res_10', 'hla_match_dqb1_high',
        'hla_match_c_low', 'hla_match_drb1_low', 'hla_match_dqb1_low'
    ]
    for col in numerical_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Handle missing values exactly as in your notebook
    df['rituximab'] = df['rituximab'].fillna(df['rituximab'].mode()[0]).map({'Yes': 1, 'No': 0})
    df['mrd_hct'] = encode_columns(df['mrd_hct'])
    df['in_vivo_tcd'] = encode_columns(df['in_vivo_tcd'])
    df['hepatic_severe'] = encode_columns(df['hepatic_severe'])
    df['prior_tumor'] = encode_columns(df['prior_tumor'])
    df['peptic_ulcer'] = encode_columns(df['peptic_ulcer'])
    df['rheum_issue'] = encode_columns(df['rheum_issue'])

    # Categorical columns that were explicitly one-hot encoded in your notebook
    categorical_cols = [
        'tbi_status', 'prim_disease_hct', 'obesity', 'prod_type',
        'ethnicity', 'race_group', 'gvhd_proph', 'tce_imm_match',
        'tce_div_match', 'donor_related', 'melphalan_dose', 'cardiac',
        'pulm_moderate'
    ]

    # Fill missing values before encoding
    for col in ['ethnicity', 'obesity']:
        if col in df.columns:
            df[col] = df[col].fillna("Missing")

    for col in ['tce_imm_match', 'tce_div_match', 'cardiac', 'pulm_moderate']:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown")

    # Apply one-hot encoding
    df = pd.get_dummies(df, columns=[col for col in categorical_cols if col in df.columns], dtype=int)

    # Ensure test dataset aligns with training columns
    if reference_columns is not None:
        for col in reference_columns:
            if col not in df.columns:
                df[col] = 0  # Add missing columns
        df = df[reference_columns]  # Ensure column order matches

    # Convert True/False values to 1/0
    df = df.applymap(lambda x: 1 if x is True else (0 if x is False else x))

    return df


train_file_path = "1oCgZEJBCPQAJc9Jc2UVJtLBejSmMlR9F"
test_file_path = "1L-XeJlshuo0UkZ902VqQWw1T224BUoxZ"
train_url = f"https://drive.google.com/uc?export=download&id={train_file_path}"
test_url = f"https://drive.google.com/uc?export=download&id={test_file_path}"
train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

train_df = preprocess_features(train_df)
final_train_columns = train_df.columns.tolist()
test_df = preprocess_features(test_df, final_train_columns)

<ipython-input-29-2a37e7eb7da1>:104: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: 1 if x is True else (0 if x is False else x))
<ipython-input-29-2a37e7eb7da1>:104: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: 1 if x is True else (0 if x is False else x))
